# 04 · Statistical Analysis

**Objective:** Apply hypothesis tests to validate EDA patterns.

| # | Test | Question |
|---|------|----------|
| 1 | Shapiro-Wilk | Are prices normally distributed? |
| 2 | Welch t-test | Do high-rated products cost more? |
| 3 | One-Way ANOVA | Do mean prices differ across categories? |
| 4 | Chi-Square | Is sponsorship independent of badge status? |
| 5 | Spearman | Does discount % correlate with reviews? |

In [ ]:
import pandas as pd, numpy as np
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed.csv')
print(f'Rows: {len(df):,}  |  Columns: {df.shape[1]}')

In [ ]:
# 1. Normality test on discounted_price
sample = df['discounted_price'].dropna().sample(5000, random_state=42)
stat, p = stats.shapiro(sample)
print(f'Shapiro-Wilk  W={stat:.6f}  p={p:.2e}')
print('Result:', 'NOT normal (reject H0)' if p < 0.05 else 'Normal')

In [ ]:
# 2. Welch t-test: high-rated (>=4.5) vs lower-rated prices
high = df.loc[df['product_rating'] >= 4.5, 'discounted_price'].dropna()
low  = df.loc[df['product_rating'] <  4.5, 'discounted_price'].dropna()
t, p = stats.ttest_ind(high, low, equal_var=False)
print(f'High n={len(high):,} mean=${high.mean():.2f}')
print(f'Low  n={len(low):,} mean=${low.mean():.2f}')
print(f't={t:.4f}  p={p:.4e}')
print('Result:', 'Significant difference' if p < 0.05 else 'No difference')

In [ ]:
# 3. ANOVA: mean price across top-5 categories
top5 = df['product_category'].value_counts().head(5).index
groups = [g['discounted_price'].dropna() for _, g in df[df['product_category'].isin(top5)].groupby('product_category')]
f, p = stats.f_oneway(*groups)
print(f'Categories: {list(top5)}')
print(f'F={f:.4f}  p={p:.4e}')
print('Result:', 'At least one category differs' if p < 0.05 else 'No difference')

In [ ]:
# 4. Chi-Square: sponsorship vs badge status
ct = pd.crosstab(df['is_sponsored'], df['is_best_seller'])
display(ct)
chi2, p, dof, _ = stats.chi2_contingency(ct)
print(f'chi2={chi2:.4f}  dof={dof}  p={p:.4e}')
print('Result:', 'NOT independent' if p < 0.05 else 'Independent')

In [ ]:
# 5. Spearman: discount_percentage vs total_reviews
sub = df[['discount_percentage','total_reviews']].dropna()
rho, p = stats.spearmanr(sub['discount_percentage'], sub['total_reviews'])
print(f'Spearman rho={rho:.4f}  p={p:.4e}')

---
## Summary
| Test | Key Finding |
|------|-------------|
| Shapiro-Wilk | Prices are **not** normally distributed |
| Welch t-test | Check output for rating-price relationship |
| ANOVA | Check output for category-level pricing |
| Chi-Square | Check output for sponsorship-badge link |
| Spearman | Check output for discount-review correlation |